# 04 - Follow-up Signal Analysis

## 项目背景

用户的追问行为包含丰富的产品信号，但不是所有追问都意味着同一件事：
- "能再详细说说吗" → 用户满意，想深入了解（**探索式追问**）
- "不是，我问的是..." → 用户不满意，在纠正模型（**纠错式追问**）

这两种追问对产品质量的含义完全相反。如果不加区分，追问率就是一个模糊指标。

## 本notebook的目标

将追问事件拆分为探索式和纠错式，分析两种追问在不同场景下的分布差异。

## 流程

**Step 1: 提取追问事件**
- 从事件表中筛选所有followup事件及其追问文本

**Step 2: 追问分类**
- 基于文本特征（关键词 + 语义信号）将追问分为探索式和纠错式

**Step 3: 分布分析**
- 整体探索式 vs 纠错式占比
- 按意图场景拆解：哪些场景纠错式追问最多

**Step 4: 追问与其他行为的关联**
- 纠错式追问后用户是否更容易点踩
- 探索式追问后用户是否更容易点赞

## 输出

追问分类结果及交叉分析图表，在notebook内展示。

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

INTENT_CN = {
    'code_generation': '代码生成', 'knowledge_qa': '知识问答',
    'creative_writing': '创意写作', 'translation': '翻译',
    'chitchat': '闲聊', 'data_analysis': '数据分析'
}

FOLLOWUP_COLORS = {
    '探索式': '#2ecc71',
    '纠错式': '#e74c3c'
}

# 加载数据
convs = pd.read_csv('../data/conversations.csv', parse_dates=['timestamp'])
events = pd.read_csv('../data/events.csv', parse_dates=['event_timestamp'])

# 提取追问事件
followups = events[events['event_type'] == 'followup'].copy()
followups = followups.merge(convs[['conversation_id', 'intent']], on='conversation_id')

print(f'追问事件总数: {len(followups)}')
print(f'有追问文本的: {(followups["followup_text"] != "").sum()}')

In [ ]:
# === 追问分类：探索式 vs 纠错式 ===

# 纠错信号关键词
CORRECTIVE_KEYWORDS = [
    '不是', '你理解错', '有问题', '不对', '你没有回答', '矛盾',
    '不是这个意思', '明显不对', '能不能直接', '太笼统'
]

def classify_followup(text):
    if any(kw in text for kw in CORRECTIVE_KEYWORDS):
        return '纠错式'
    return '探索式'

followups['followup_type'] = followups['followup_text'].apply(classify_followup)

print('追问分类结果:')
counts = followups['followup_type'].value_counts()
for t, c in counts.items():
    print(f'  {t}: {c} 条 ({c/len(followups)*100:.1f}%)')

In [ ]:
# === 按意图场景拆解追问类型 ===

cross_followup = pd.crosstab(followups['intent'], followups['followup_type'], normalize='index') * 100
cross_followup = cross_followup.round(1)
cross_followup = cross_followup.rename(index=INTENT_CN)

# 按纠错式占比降序
cross_followup = cross_followup.sort_values('纠错式', ascending=False)

fig = go.Figure()

for ftype in ['探索式', '纠错式']:
    fig.add_trace(go.Bar(
        name=ftype,
        x=cross_followup.index,
        y=cross_followup[ftype],
        marker_color=FOLLOWUP_COLORS[ftype],
        text=[f'{v:.0f}%' for v in cross_followup[ftype]],
        textposition='inside',
        textfont=dict(size=12)
    ))

fig.update_layout(
    barmode='stack',
    title='各意图场景追问类型分布',
    xaxis_title='意图场景',
    yaxis_title='百分比 (%)',
    template='plotly_white',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.show()

print('\n各场景纠错式追问占比:')
print(cross_followup[['纠错式']].to_string())

In [ ]:
# === 追问后的二次行为关联 ===

# 找出追问后有二次行为的对话
followup_events = events[events['event_type'] == 'followup'][['conversation_id', 'event_timestamp']].copy()
followup_events.columns = ['conversation_id', 'followup_ts']

# 找追问之后的下一个事件
all_events = events.merge(followup_events, on='conversation_id', how='inner')
after_followup = all_events[all_events['event_timestamp'] > all_events['followup_ts']]
next_action = after_followup.sort_values('event_timestamp').groupby('conversation_id').first().reset_index()

# 关联追问类型
next_action = next_action.merge(
    followups[['conversation_id', 'followup_type']],
    on='conversation_id'
)

# 统计
cross_next = pd.crosstab(next_action['followup_type'], next_action['event_type'], normalize='index') * 100
cross_next = cross_next.round(1)

fig = go.Figure()

action_colors = {'like': '#2ecc71', 'dislike': '#e74c3c'}
action_cn = {'like': '点赞', 'dislike': '点踩'}

for action in ['like', 'dislike']:
    if action in cross_next.columns:
        fig.add_trace(go.Bar(
            name=action_cn[action],
            x=cross_next.index,
            y=cross_next[action],
            marker_color=action_colors[action],
            text=[f'{v:.1f}%' for v in cross_next[action]],
            textposition='outside',
            textfont=dict(size=13)
        ))

fig.update_layout(
    barmode='group',
    title='追问后的二次行为：探索式 vs 纠错式',
    xaxis_title='追问类型',
    yaxis_title='百分比 (%)',
    template='plotly_white',
    height=450,
    yaxis=dict(range=[0, cross_next.max().max() * 1.3]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.show()

print('\n追问后二次行为分布:')
print(cross_next.to_string())

## 关键发现

**1. 追问类型分布**
- 整体：探索式59.8%，纠错式40.2%
- 如果不加区分，追问率是一个模糊指标；拆开后才能看到真实的产品信号

**2. 各场景纠错式追问差异显著**
- 知识问答纠错式占比最高（64%），与03中幻觉问题高度吻合——用户在纠正模型的错误事实
- 数据分析（43%）和代码生成（39%）紧随其后，用户在纠正模型对需求的理解偏差
- 创意写作和闲聊纠错式极低（~10%），追问多为正向的深入探索

**3. 追问后二次行为**
- 探索式和纠错式追问后的点赞/点踩比例差异不显著（约50:50）
- 这是合成数据的局限——01中事件流生成逻辑未充分关联追问类型与后续行为。真实数据中预期纠错式追问后点踩率应显著更高

**4. 产品启示**
- 追问率不应作为单一指标，需拆分为探索式和纠错式分别监控
- 知识问答场景的高纠错率（64%）进一步印证了需要优先解决幻觉问题
- 纠错式追问可作为自动化质量监控的信号源——检测到纠错式追问时自动标记该对话为潜在bad case

> 下一步：05 汇总全部分析发现，输出可执行的产品优化建议